In [ ]:
import time

import astropy.units as u
import batoid
import numpy as np
from astropy.coordinates import Angle
from StarSharp.datatypes.state import StateFactory
from batoid_rubin import LSSTBuilder
from lsst.obs.lsst import LsstCam

from StarSharp.models.raytraced import RaytracedOpticalModel
from StarSharp.models.linear import LinearModel
from StarSharp import State

import matplotlib.pyplot as plt

In [ ]:
# Build a raytraced model
telescope = batoid.Optic.fromYaml("LSST_r.yaml")
builder = LSSTBuilder(
    telescope,
    # Set coords/units to match MTAOS
    dof_coord_system="OCS",
    flip_m2_bending_modes=False,
    dof_angle_units="degree"
)
camera = LsstCam().getCamera()
model = RaytracedOpticalModel(
    builder,
    rtp = Angle("0 deg"),
    wavelength=620 * u.nm,
    camera=camera
)
fc = model.make_ccd_field(detnums=list(range(4, 189, 9))) # Raft centers

In [ ]:
# Got field coordinates in both OCS and CCS:
fig, axs = plt.subplots(ncols=2, figsize=(7, 3.5))
axs[0].set_title("OCS")
axs[0].set_xlabel("x (mm)")
axs[0].set_ylabel("y (mm)")
axs[1].set_title("CCS")
axs[1].set_xlabel("x (mm)")
axs[1].set_ylabel("y (mm)")
axs[0].scatter(fc.ocs.x, fc.ocs.y)
axs[1].scatter(fc.ccs.x, fc.ccs.y)
plt.tight_layout()
plt.show()

In [ ]:
use_dof = list(range(17))+list(range(30,37))

# Get spots
steps = model.steps[use_dof] # Default differential step sizes for each DOF
rng = np.random.default_rng(12345)
state = State(
    value=rng.normal(0.0, size=24) * steps,
    basis="x",
    use_dof=use_dof,
    n_dof=50
)
mspots = model.spots(fc, state, reference="ring")

fig, axs = plt.subplots(nrows=5, ncols=5, figsize=(10, 10))
for ax, mspot in zip(axs.flat, mspots):
    ax.scatter(
        mspot.dx[~mspot.vignetted], mspot.dy[~mspot.vignetted],
        color="C0", label="Raytraced", s=1
    )
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axs[4,1:]:
    fig.delaxes(ax)
plt.legend()
plt.show()

In [ ]:
t0 = time.time()
rng = np.random.default_rng(123)
for _ in range(50):
    state = State(
        value=rng.normal(0.0, size=24) * steps,
        basis="x",
        use_dof=use_dof,
        n_dof=50
    )
    _ = model.spots(fc, state)
t1 = time.time()
print(f"Time for 50 iterations: {t1 - t0} seconds")

In [ ]:
# Linearized model
linear = LinearModel(
    raytraced=model,
    # Have to declare the field and use_dof up front
    field=fc,
    use_dof=use_dof,
    # Other args to declare up front
    spots_kwargs=dict(reference="ring"),
    zernikes_kwargs=dict(algorithm="ta", reference="ring", include_chip_heights=False)
)

# First execution builds the sensitivity matrix internally,
# should be fast after that.
rng = np.random.default_rng(12345)
state = State(
    value=rng.normal(0.0, size=24) * steps,
    basis="x",
    use_dof=use_dof,
    n_dof=50
)

lspots = linear.spots(state)

fig, axs = plt.subplots(nrows=5, ncols=5, figsize=(10, 10))
for ax, mspot, lspot in zip(axs.flat, mspots, lspots):
    ax.scatter(
        mspot.dx[~mspot.vignetted], mspot.dy[~mspot.vignetted],
        color="b", label="Raytraced", s=10
    )
    ax.scatter(
        lspot.dx[~lspot.vignetted], lspot.dy[~lspot.vignetted],
        color="r", label="Linear", s=1
    )
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axs[4,1:]:
    fig.delaxes(ax)
plt.legend()
plt.show()

In [ ]:
t0 = time.time()
rng = np.random.default_rng(123)
for _ in range(50):
    state = State(
        value=rng.normal(0.0, size=24) * steps,
        basis="x",
        use_dof=use_dof,
        n_dof=50
    )
    _ = linear.spots(state)
t1 = time.time()
print(f"Time for 50 iterations: {t1 - t0} seconds")

In [ ]:
# Compare moments
m2 = mspots.moments()
l2 = lspots.moments()
for i in range(5):
    print()
    print(f"{m2.xx[i]:.3f} vs {l2.xx[i]:.3f}")
    print(f"{m2.yy[i]:.3f} vs {l2.yy[i]:.3f}")
    print(f"{m2.T[i]:.3f} vs {l2.T[i]:.3f}")
    print(f"{m2.e1[i]:.3f} vs {l2.e1[i]:.3f}")
    print(f"{m2.e2[i]:.3f} vs {l2.e2[i]:.3f}")

In [ ]:
state.x

In [ ]:
mzk = model.zernikes(fc, state*10, algorithm="ta", reference="ring", include_chip_heights=False)
lzk = linear.zernikes(state*10)

In [ ]:
(lzk.coefs-mzk.coefs).to_value(u.nm)[:, 4:].max()

In [ ]:
t0 = time.time()
rng = np.random.default_rng(123)
for _ in range(50):
    state = State(
        value=rng.normal(0.0, size=24) * steps,
        basis="x",
        use_dof=use_dof,
        n_dof=50
    )
    _ = model.zernikes(fc, state, algorithm="ta", reference="ring")
t1 = time.time()
print(f"Time for 50 iterations: {t1 - t0} seconds")

In [ ]:
t0 = time.time()
rng = np.random.default_rng(123)
for _ in range(50):
    state = State(
        value=rng.normal(0.0, size=24) * steps,
        basis="x",
        use_dof=use_dof,
        n_dof=50
    )
    _ = linear.zernikes(state)
t1 = time.time()
print(f"Time for 50 iterations: {t1 - t0} seconds")